In [ ]:
from gitsource import GithubRepositoryDataReader

In [ ]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

len(documents)

In [ ]:
import minsearch

index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

query = "How does the agentic loop keep calling the model until it stops?"

results = index.search(query, num_results=5)

results[0]["filename"]

In [ ]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

len(chunks)

In [ ]:
import os

os.environ["OPENAI_API_KEY"] = "PASTE_YOUR_KEY_HERE"

In [ ]:
from openai import OpenAI

client = OpenAI()

def build_context(results):
    context = ""

    for doc in results:
        context += f"filename: {doc['filename']}\n"
        context += f"content: {doc['content']}\n\n"

    return context


def rag(query, index, num_results=5):
    results = index.search(query, num_results=num_results)
    context = build_context(results)

    prompt = f"""
You're a course teaching assistant. Answer the question based only on the context below.

Context:
{context}

Question:
{query}
""".strip()

    response = client.responses.create(
        model="gpt-5.4-mini",
        input=prompt,
    )

    return response.output_text, response.usage

In [ ]:
answer_q3, usage_q3 = rag(query, index)

print(answer_q3)
print(usage_q3.input_tokens)

In [ ]:
chunk_index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

chunk_index.fit(chunks)

In [ ]:
answer_q5, usage_q5 = rag(query, chunk_index)

print(answer_q5)
print(usage_q5.input_tokens)

In [ ]:
print("Q3 tokens:", usage_q3.input_tokens)
print("Q5 tokens:", usage_q5.input_tokens)
print("Ratio:", usage_q3.input_tokens / usage_q5.input_tokens)

In [ ]:
search_calls = 0

def search(query: str) -> list[dict]:
    """
    Search the course lesson chunks for information relevant to the query.
    """
    global search_calls
    search_calls += 1

    results = chunk_index.search(query, num_results=5)

    return [
        {
            "filename": doc["filename"],
            "content": doc["content"],
        }
        for doc in results
    ]

In [ ]:
from openai import OpenAI
import json

client = OpenAI()

tools = [
    {
        "type": "function",
        "name": "search",
        "description": "Search the course lesson chunks for information relevant to the query.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query",
                }
            },
            "required": ["query"],
            "additionalProperties": False,
        },
    }
]

instructions = """
You're a course teaching assistant. Answer the student's question using the search tool.
Make multiple searches with different keywords before answering.
""".strip()

question = "How does the agentic loop work, and how is it different from plain RAG?"

input_messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question},
]

search_calls = 0

while True:
    response = client.responses.create(
        model="gpt-5.4-mini",
        input=input_messages,
        tools=tools,
    )

    input_messages += response.output

    function_calls = [
        item for item in response.output
        if item.type == "function_call"
    ]

    if not function_calls:
        print(response.output_text)
        break

    for call in function_calls:
        args = json.loads(call.arguments)
        result = search(args["query"])

        input_messages.append({
            "type": "function_call_output",
            "call_id": call.call_id,
            "output": json.dumps(result),
        })

print("Search calls:", search_calls)